# Matelda Pipeline

As data-driven applications gain popularity, ensuring high data quality is a growing concern. This requirement involves not only the quality of primary data sources but also external data sources used for data enrichment purposes. Yet, data cleaning techniques are limited to treating one table at a time. A table-by-table application of such methods is cumbersome, because these methods either require previous knowledge about constraints or often require labor-intensive configurations and manual labeling for each individual table. As a result, they hardly scale beyond a few tables and miss the chance for optimizing the cleaning process. To tackle these issues, we introduce a novel semi-supervised error detection approach, Matelda, that organizes a given set of tables by folding their cells with regard to domain and quality similarity to facilitate user supervision. The idea is to identify groups of data cells across all tables that can benefit from the same user label. For this purpose, we identify a feature embedding that makes cell values comparable across many different tables. Experimental evaluations demonstrate that Matelda outperforms various configurations of existing single-table cleaning methodologies in cleaning multiple tables at a time, in particular when the ratio of labeling budget to number of tables is very low.

For more information about Matelda, we recommend you to read the [Paper](https://openproceedings.org/2025/conf/edbt/paper-98.pdf) and view the corresponding Code on [GitHub](https://github.com/LUH-DBS/Matelda).

# Initialization of Matelda

The initialization function in Matelda sets up the necessary configurations, directories, and variables, including paths for input data, output, logs, and results. It prepares the environment for the error detection process, ensuring all required directories exist and the configuration parameters are loaded correctly.

In [1]:
import os
import multiprocessing
import pickle
import pandas as pd
from configparser import ConfigParser

import ipywidgets as widgets
from IPython.display import display, Javascript, clear_output, HTML

from pipeline_functions import init
from pipeline_functions import domain_based_folding, quality_based_folding
from config_ui import get_config_widget, create_table_content_widget, create_cell_fold_accordion

from marshmallow_pipeline.utils.loading_results import loading_columns_grouping_results

In [2]:
!conda install ipywidgets -y

Channels:
 - defaults
 - conda-forge
Platform: linux-64
Solving environment: done

# All requested packages already installed.



In [3]:
# Define the path to your config.ini file
config_path = os.path.join(os.getcwd(), "config.ini")

# Create and load the configuration
default_configs = ConfigParser()
default_configs.read(config_path)

# Build the configuration widget using our separate module function
config_widget = get_config_widget(default_configs)
# Later (in a subsequent cell) you can retrieve the updated configuration:
# updated_config = config_widget.updated_config

# Display the widget
display(config_widget)

<IPython.core.display.Javascript object>

In [10]:
# Use updated configuration or the default configuration else
if config_widget.updated_config:
    print("use updated configs")
    updated_config = config_widget.updated_config
else: 
    print("use default configs")
    updated_config = default_configs

# Initialize
execution = 0
updated_config = init(updated_config, execution)

use updated configs


# Domain-Based Cell Folding
Domain-based cell folding in Matelda organizes cells from different tables by their semantic similarities.

In [11]:
n_cores = int(updated_config["n_cores"])
pool = multiprocessing.Pool(n_cores)

# Call Domain Based Folding
table_grouping_dict, table_size_dict = domain_based_folding(updated_config, pool)

I need at least 2 labeled cells per table group to work at all and at least 2 * 6 labeled cells per table group to work effectively! Thant means you need to label 60 cells if you want reasonable (!) results:


### Results from Domain-Based Cell Folding

In [12]:
# Present the results in an interactive Widget
outer_acc = create_cell_fold_accordion(updated_config)
display(outer_acc)

Accordion(children=(Accordion(children=(Output(),), titles=('flights',)), Accordion(children=(Output(),), titl…

# Quality-Based Cell Folding

In [13]:
# Use the keys returned by init (flattened names, not nested dictionaries)
cg_enabled = updated_config["column_grouping_enabled"]
column_groups_dir = os.path.join(updated_config["mediate_files_path"], "col_grouping_res", "col_df_res")
expected_file = os.path.join(column_groups_dir, "col_df_labels_cluster_0.pickle")

if cg_enabled and os.path.exists(expected_file):
    # Load the column grouping results to get the cluster sizes and the path to the column groups file.
    # This function returns a tuple of (number_of_col_clusters, cluster_sizes_dict, column_groups_df_path)

    _, cluster_sizes_dict, column_groups_df_path = loading_columns_grouping_results(
        table_grouping_dict, updated_config["mediate_files_path"]
    )

    results = quality_based_folding(updated_config, pool, column_groups_df_path, cluster_sizes_dict)
    
    # Unpack the results
    (
        y_test_all, 
        y_local_cell_ids, 
        predicted_all, 
        y_labeled_by_user_all,
        unique_cells_local_index_collection, 
        samples, 
        n_user_labeled_cells
    ) = results
    
    # Present the results in the notebook
    print("Number of user labeled cells (quality based folding):", n_user_labeled_cells)


InvalidParameterError: The 'n_clusters' parameter of MiniBatchKMeans must be an int in the range [1, inf). Got 0 instead.

---
# Below: Old Version
--- 

In [ ]:
# Use the keys returned by init (flattened names, not nested dictionaries)
cg_enabled = updated_config["column_grouping_enabled"]
column_groups_dir = os.path.join(updated_config["mediate_files_path"], "col_grouping_res", "col_df_res")
expected_file = os.path.join(column_groups_dir, "col_df_labels_cluster_0.pickle")

if cg_enabled and os.path.exists(expected_file):
    # Load the column grouping results to get the cluster sizes and the path to the column groups file.
    # This function returns a tuple of (number_of_col_clusters, cluster_sizes_dict, column_groups_df_path)
    _, cluster_sizes_dict, column_groups_df_path = loading_columns_grouping_results(
        table_grouping_dict, updated_config["mediate_files_path"]
    )

# Assuming column_groups_df_path and cluster_sizes_dict have been set by your domain-based folding cell:
if column_groups_df_path is not None:
    results = quality_based_folding(updated_config, pool, column_groups_df_path, cluster_sizes_dict)
    
    # Unpack the results
    (
        y_test_all, 
        y_local_cell_ids, 
        predicted_all, 
        y_labeled_by_user_all,
        unique_cells_local_index_collection, 
        samples, 
        n_user_labeled_cells
    ) = results
    
    # Present the results in the notebook
    print("Number of user labeled cells (quality based folding):", n_user_labeled_cells)
else:
    print("Skipping quality based folding due to missing column grouping results.")


NameError: name 'column_groups_df_path' is not defined

In [ ]:
    results = quality_based_folding(updated_config, pool, column_groups_df_path, cluster_sizes_dict)
    
    # Unpack the results
    (
        y_test_all, 
        y_local_cell_ids, 
        predicted_all, 
        y_labeled_by_user_all,
        unique_cells_local_index_collection, 
        samples, 
        n_user_labeled_cells
    ) = results
    
    # Present the results in the notebook
    print("Number of user labeled cells (quality based folding):", n_user_labeled_cells)
else:
    print("Skipping quality based folding due to missing column grouping results.")


### Presenting results from Quality-Based Cell Folding to user

In [ ]:
###############
# Problem:    #
###############
# - we discussed last time that I should look into the pickle files which are generated in error_detection
# - not sure what to visualize and what each column is
###############

# Load the DataFrame from your pickle file
df = pd.read_pickle('/home/julian/projects/Matelda/output_quintet/output_quintet_0/_test_edbt_Quintet_27700_labels/cell_clustering/all_cell_clusters_records.pickle')
#df = pd.read_pickle('/home/julian/projects/Matelda/output_quintet/output_quintet_0/_test_edbt_Quintet_27700_labels/cell_clustering/cell_cluster_cells_dict_all.pickle')
#columns_to_show = ['table_cluster', 'col_cluster', 'n_cells', 'cells_per_cluster']
#df_sample = df[columns_to_show].head(5)
df_sample = df.head(5)
df_sample

In [ ]:
####################
# Test             #
####################
# - tested the ipywidgets expand/collapse buttons
# - but thats probably not what we want to visualize
####################

# Load the DataFrame from your pickle file.
df = pd.read_pickle('/home/julian/projects/Matelda/output_quintet/output_quintet_0/_test_edbt_Quintet_27700_labels/cell_clustering/all_cell_clusters_records.pickle')

# Choose the columns you want to display and take a small sample.
columns_to_show = ['table_cluster', 'col_cluster', 'n_cells', 'cells_per_cluster']
df_sample = df[columns_to_show].head(5).copy()  # Using head(5) to keep it light.

def format_cells_per_cluster(cell_cluster):
    """
    Convert the cells_per_cluster dictionary into a multi-line string.
    For each key, show only the first 10 items of the list (if the list is long).
    """
    if isinstance(cell_cluster, dict):
        lines = []
        for key, value in cell_cluster.items():
            # Show a preview of the list; adjust the number (10 here) as needed.
            if isinstance(value, list) and len(value) > 10:
                preview = value[:10]
                lines.append(f"{key}: {preview} ... ({len(value)} items)")
            else:
                lines.append(f"{key}: {value}")
        return "\n".join(lines)
    return str(cell_cluster)

# Create an accordion widget for each row in the sample.
accordion_items = []
for idx, row in df_sample.iterrows():
    # Build a summary header for the accordion.
    header = f"Cluster: ({row['table_cluster']}, {row['col_cluster']}), n_cells: {row['n_cells']}"
    
    # Format the full details of cells_per_cluster.
    details = format_cells_per_cluster(row['cells_per_cluster'])
    
    # Create an output widget to hold the formatted details (using <pre> to preserve formatting).
    detail_output = widgets.HTML(value=f"<pre>{details}</pre>")
    
    # Create an Accordion widget with this detail.
    accordion = widgets.Accordion(children=[detail_output])
    accordion.set_title(0, header)
    
    accordion_items.append(accordion)

# Display all the accordions vertically.
display(widgets.VBox(accordion_items))